# FQ Clustering Benchmark

Run this workflow to reproduce the algorithm benchmark with package imports instead of copied utility code.

## 1. Configuration

Set sample count, number of experiments, and resolutions. Environment variables `NSIM`, `N_EXP`, and `R_LIST` are honored for CI smoke runs.

In [ ]:
import sys
from pathlib import Path
for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "fq").exists():
        sys.path.insert(0, str(candidate))
        break
import ast, os
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

from fq.algorithms import ALGORITHMS
from fq.data import synthesize_paper_data
from fq.runner import build_master_df, run_algorithm
from fq import plots

NSIM = int(os.environ.get("NSIM", "200"))
N_EXP = int(os.environ.get("N_EXP", "3"))
R_LIST = ast.literal_eval(os.environ.get("R_LIST", "[128, 256, 512]"))
N_QUANTA = int(os.environ.get("N_QUANTA", "10"))
MAX_ITER = int(os.environ.get("MAX_ITER", "10"))
TOL = 1e-3
RESULTS_DIR = Path("results/runtime/notebook")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print(NSIM, N_EXP, R_LIST)

## 2. Data and Fig. 2

Generate the paper fallback lognormal random field from the $S^1_{FF}(\omega)$ spectrum.

In [ ]:
X = synthesize_paper_data(nsim=NSIM, R=max(R_LIST), seed=20260120)
fig, ax = plt.subplots(figsize=(7, 3))
for row in X[:5]:
    ax.plot(row, alpha=0.8)
ax.set_title("Fig. 2 style: sample random-field realizations")
ax.set_xlabel("Grid index"); ax.set_ylabel("Field value")
plt.show()

## 3. Algorithms

LX uses exhaustive Lloyd updates with random initialization and complexity $O(N_{sim} N R n_{iter})$. LK uses k-means++ initialization with the same Lloyd refinement. KN is a one-pass nearest-neighbor quantizer with complexity $O(N_{sim}NR)$. MB uses MiniBatch k-means as a fast approximate CVT method.

In [ ]:
METHODS = [m for m in ["LX", "LK", "KN", "MB"] if m in ALGORITHMS]
results = {}
for name in METHODS:
    alg = ALGORITHMS[name](n_quanta=N_QUANTA, max_iter=MAX_ITER, tol=TOL)
    results[name] = run_algorithm(alg, R_LIST, X, n_exp=N_EXP, store_centers_for_R=R_LIST[0], results_dir=RESULTS_DIR, verbose_every=0)
summary = build_master_df(results)
summary

## Results

Load checkpoints or use the in-memory results, then call the paper-style figure functions.

In [ ]:
fig_dir = RESULTS_DIR / "figures"
plots.fig6_mean_time(results, save_path=fig_dir / "fig6_mean_time.png")
plots.fig3_RD_boxplots(results, save_path=fig_dir / "fig3_RD_boxplots.png")
plots.fig4_RD_trend(results, save_path=fig_dir / "fig4_RD_trend.png")
plots.fig7_complexity(results, save_path=fig_dir / "fig7_complexity.png")
plots.fig8_iter_cvt(results, save_path=fig_dir / "fig8_iter_cvt.png")
plots.fig9_accuracy(results, X, R_LIST[0], save_path=fig_dir / "fig9_accuracy.png")
plots.pareto(results, R_values=R_LIST, save_path=fig_dir / "pareto.png")
print(fig_dir)